# Solutions · Assessment · Module 03

Mark **Part A** and **Part C** against these. Part B marked itself in the assessment notebook; the
values are printed below so you can see where an answer went wrong rather than only that it did.

**Marking Part A:** one mark each, awarded for the substance, not the wording. Where an answer has two
required halves the mark is all-or-nothing - both halves or neither. That is deliberate: half these
questions are ones people can half-answer while still holding the misconception.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# SYNTHETIC. One row per parcel delivered from a two-depot courier network, one month.
def load_parcels():
    rng = np.random.default_rng(3003)
    n = 400
    depot = rng.choice(["north", "south"], n, p=[0.55, 0.45])
    distance_km = np.round(rng.uniform(1.0, 40.0, n), 1)
    weight_kg = np.round(np.exp(rng.normal(0.6, 0.75, n)), 2)
    minutes = np.round(12 + 1.8 * distance_km
                       + np.where(depot == "north", 4.0, 0.0)
                       + rng.normal(0, 6.0, n), 1)
    damaged = (rng.random(n) < 0.06).astype(int)
    hit = np.where(damaged == 1, 0.85, 0.09)
    scanned = (rng.random(n) < hit).astype(int)
    return pd.DataFrame({"parcel_id": np.arange(1, n + 1), "depot": depot,
                         "distance_km": distance_km, "weight_kg": weight_kg,
                         "minutes": minutes, "scanned": scanned, "damaged": damaged})


EXPECTED = {
    "B1": "1663eb5c7d11", "B2": "8af6bd78ca81", "B3": "bc8ceb912fcd", "B4": "86f66c6599bc",
    "B5": "8928a0f09e20", "B6": "be31944da50d", "B7": "d34a97dda0c7", "B8": "2bf63beea2d0",
}


def check(task, answer):
    # Marks one Part B answer without revealing it. Counts: whole numbers. Everything else: 2 dp.
    task = task.upper()
    if task not in EXPECTED:
        print("unknown task:", task)
        return
    candidates = [answer] if isinstance(answer, (int, np.integer)) else [
        round(float(answer) + delta, 2) for delta in (-0.01, 0.0, 0.01)
    ]
    for value in candidates:
        text = str(int(value)) if isinstance(answer, (int, np.integer)) else "%.2f" % value
        if hashlib.sha256((task + "|" + text).encode()).hexdigest()[:12] == EXPECTED[task]:
            print("%s  correct" % task)
            return
    print("%s  not yet - check your working, then try again" % task)


parcels = load_parcels()
print("loaded %d rows, %d columns" % parcels.shape)
print(parcels.head(3).to_string(index=False))

## Part A · Recall

**A1.** The **mean** minimises the total **squared** error; the **median** minimises the total
**absolute** error. A long right tail pulls the **mean** further, because a single distant value
contributes its distance *squared*, while the median only counts how many values sit on each side.

*Mark: both minimisers and the right answer to the tail.*

**A2.** `ddof=1` divides by `n - 1` instead of `n`, which makes the result larger. It is there because
the deviations were measured from the **sample's own mean**, which sits closer to the sample than the
true mean does, so the spread comes out systematically too small; dividing by `n - 1` corrects that bias.

*Mark: the divisor change and any correct statement of why.*

**A3.** **1,600 rows.** The standard error falls with the square root of the sample size, so halving it
needs **four times** as much data. (Exactly 1,637 with the actual standard deviation - see B3 - but the
rule is what is being marked.)

*Mark: four-times, or the square-root rule stated.*

**A4.** It means: **if you repeated the whole sampling procedure many times, 95% of the intervals
constructed this way would contain the true value.** The confidence belongs to the procedure, not to any
one interval.

What it does **not** mean: that there is a 95% probability the true value lies inside *this* particular
interval; that 95% of the data lies inside it; or that a value just outside it has been ruled out.

*Mark: a correct long-run statement plus one correct disclaimer.*

**A5.** Any example with a large asymmetry, for instance: **P(the animal has four legs given it is a
dog)** is essentially 1, while **P(it is a dog given it has four legs)** is small. Or: almost every
person who drowns had eaten in the previous day; almost nobody who eats drowns.

People almost always have the **more useful, rarer conditional** in mind - "given the evidence, what is
the state of the world" - while the number that is easy to measure is the other one, "given the state of
the world, how often do we see this evidence". Confusing the two is the base-rate fallacy.

*Mark: a genuinely asymmetric example plus naming the direction people intend.*

**A6.** 1. **Start with a concrete population** - say 1,000 cases - and split it by the base rate.
2. **Apply the test's hit rates to each group separately**, giving four counts.
3. **Read the answer off the counts:** of everyone who tested positive, what fraction actually has the
condition.

*Mark: all three steps, in an order that works.*

**A7.** **Minutes per kilometre.** One more kilometre of distance is associated with 1.8 more minutes.

If the outcome had been `log(minutes)`, a coefficient of 1.8 would be a **multiplicative** effect: each
extra kilometre multiplies the time by `exp(1.8)`, about 6-fold - which for this data would be absurd,
and that is the point. On a logged outcome, small coefficients are read as approximate percentages
(0.05 is about +5% per unit), and large ones must be exponentiated.

*Mark: the units, and multiplicative-not-additive.*

**A8.** Because the squared differences are computed in each column's **own units**, and a column whose
numbers are larger produces larger differences, which dominate the sum once squared. It is the column
with the **largest numerical spread**, which has nothing to do with the column being important.

*Mark: units/spread, not importance.*

**A9.** The **sign** says which way to move: negative means increasing the parameter reduces the loss, so
step up. The **size** says how far you are from the bottom - the gradient shrinks towards zero as the
bottom approaches, which is why the steps get smaller on their own without anybody shrinking them.

*Mark: both, and they must be different things.*

**A10.** **Too small:** the loss falls but the curve is still visibly descending at the last step -
correct direction, never arrives. **Too large:** the loss rises, oscillates, or becomes `nan` - each step
overshoots the bottom and lands further up the opposite side, so the next step is longer still.

*Mark: both, with a distinguishable appearance for each.*

## Part B · The values, and where each goes wrong

In [ ]:
work = parcels.copy()
work["late"] = (work["minutes"] > 45).astype(int)

# B1
b1 = work.weight_kg.mean() - work.weight_kg.median()
print("B1  mean %.4f - median %.4f = %.2f" % (work.weight_kg.mean(), work.weight_kg.median(), b1))

# B2
spread = work.minutes.std(ddof=1)
b2 = spread / np.sqrt(len(work))
print("B2  sd(ddof=1) %.4f / sqrt(%d) = %.2f" % (spread, len(work), b2))

# B3
b3 = int(np.ceil((spread / 0.50) ** 2))
print("B3  (%.4f / 0.50)^2 = %.1f -> %d rows" % (spread, (spread / 0.50) ** 2, b3))

# B4
north = work[work.depot == "north"]
b4 = 100 * north.late.mean()
print("B4  %d of %d north parcels late = %.2f%%" % (north.late.sum(), len(north), b4))

# B5
flagged = work[work.scanned == 1]
b5 = 100 * flagged.damaged.mean()
print("B5  %d of %d flagged parcels damaged = %.2f%%" % (flagged.damaged.sum(), len(flagged), b5))

# B6 and B8
slope, intercept = np.polyfit(work.distance_km, work.minutes, 1)
b8 = np.mean((work.minutes - (intercept + slope * work.distance_km)) ** 2)
print("B6  slope %.2f minutes per km (intercept %.2f)" % (slope, intercept))
print("B8  mean squared error %.2f" % b8)

# B7
pair = work[["distance_km", "weight_kg"]].to_numpy()
standardised = (pair - pair.mean(axis=0)) / pair.std(axis=0)
b7 = np.sqrt(((standardised[0] - standardised[1]) ** 2).sum())
print("B7  standardised distance between rows 1 and 2 = %.2f" % b7)
print("    (raw, for comparison: %.2f - almost all of it the kilometres)"
      % np.sqrt(((pair[0] - pair[1]) ** 2).sum()))

**Where each one goes wrong.**

- **B1** - using `.mean()` on the wrong column, or reporting the two numbers rather than the difference.
  The gap is positive because the weights are right-skewed; if you got a negative number you subtracted
  the wrong way round.
- **B2** - forgetting `ddof=1`, or dividing by `n` instead of `sqrt(n)`. Dividing by 400 rather than 20
  gives 0.05, which is a factor of twenty out and the commonest miss on this task.
- **B3** - answering 1,600. That is right for the *rule* and wrong for *this data*: with a standard
  deviation of 20.2293 the exact requirement is 1,637. Both were accepted in spirit, only one by the
  checker. If you answered 800 you halved the rows instead of quadrupling them.
- **B4** - defining late as `>= 45` rather than `> 45`, or computing the percentage over all parcels
  instead of over north ones.
- **B5** - answering 85 (the scanner's hit rate on damaged parcels) or 6 (the base rate). Both are real
  numbers in this data and neither is the question. The question is P(damaged given flagged).
- **B6** - fitting `distance_km` against `minutes` the wrong way round, which gives 0.51.
- **B7** - standardising with `ddof=1`, which shifts the answer slightly, or forgetting to standardise
  at all - the raw distance is 26.30 and is essentially the kilometre difference alone.
- **B8** - computing the root mean squared error (6.03) instead of the mean squared error.

## Part C · Judgement

### C1 (5 marks) · Read the picture

In [ ]:
target = parcels["minutes"].to_numpy()
feature = parcels["distance_km"].to_numpy()
feature = (feature - feature.mean()) / feature.std()

best = np.polyfit(feature, target, 1)
floor = np.mean((target - (best[1] + best[0] * feature)) ** 2)


def descend(learning_rate, steps=60):
    slope, intercept, history = 0.0, 0.0, []
    with np.errstate(over="ignore", invalid="ignore"):
        for _ in range(steps):
            residual = (intercept + slope * feature) - target
            slope -= learning_rate * 2 * np.mean(residual * feature)
            intercept -= learning_rate * 2 * np.mean(residual)
            history.append(np.mean((target - (intercept + slope * feature)) ** 2))
    return np.array(history)


print("the best achievable loss is %.2f" % floor)
print()
for label, rate in [("A", 0.0005), ("B", 0.05), ("C", 0.50), ("D", 1.02)]:
    curve = descend(rate)
    print("curve %s  rate %.4f   loss at step 1 %12.2f   at step 60 %12.2f   %s"
          % (label, rate, curve[0], curve[-1],
             "diverging" if curve[-1] > curve[0] else
             "arrived" if curve[-1] < floor * 1.01 else "still descending"))

**1. Which is which.**

| Curve | Rate | Verdict |
|---|---|---|
| **A** | 0.0005 | **far too small** - it has removed 11% of the loss in 60 steps and looks nearly flat on the log scale, but it is flat *high up*, at 2,643 against an achievable 36.33 |
| **B** | 0.05 | **sensible** - a clean descent that flattens onto the floor by about step 45 |
| **C** | 0.50 | **fastest that works** - it is already at the bottom by the first plotted step, because on a standardised feature `1/2` is exactly the ideal step |
| **D** | 1.02 | **diverging** - rising steadily, a straight line upward on a log scale, which is what exponential growth looks like |

**2. Which would you ship.** **B, at 0.05.** C arrives faster but sits on the edge: the safe range here
ends just below 1.0, so 0.5 is halfway to divergence, and that margin exists only because this feature
was standardised and there is exactly one of it. Add a second feature on a different scale, or a slightly
different dataset next month, and 0.5 can cross the line while 0.05 has an order of magnitude of room.

The general form of the argument: **a hyperparameter tuned to the edge of stability is not tuned, it is
gambled.** You are choosing between 40 steps and 1 step of a computation that takes microseconds, and
paying for the difference with the risk that the pipeline fails silently on data you have not seen yet.

*Marks: 2 for the table, 2 for choosing B with a stability argument, 1 for part 3.*

**3. Why "the loss is going down" is not enough.** Curve A is falling monotonically at every step and is
still nearly two orders of magnitude away from the achievable loss. A descending curve tells you the
direction is right and says nothing at all about whether you have arrived - you learn that only by
comparing against something, either a known floor or the curve visibly flattening.

### C2 (5 marks) · Act on the flag?

In [ ]:
table = pd.crosstab(parcels.damaged, parcels.scanned,
                    rownames=["damaged"], colnames=["scanned"])
print(table.to_string())
print()
flagged = parcels[parcels.scanned == 1]
damaged = parcels[parcels.damaged == 1]
print("of %d flagged parcels, %d are damaged and %d are not"
      % (len(flagged), flagged.damaged.sum(), (flagged.damaged == 0).sum()))
print("of %d damaged parcels, %d are caught and %d are missed"
      % (len(damaged), damaged.scanned.sum(), (damaged.scanned == 0).sum()))
print()
print("sensitivity (caught | damaged)     %.2f%%" % (100 * damaged.scanned.mean()))
print("specificity (not flagged | clean)  %.2f%%" % (100 * (1 - parcels[parcels.damaged == 0].scanned.mean())))
print("precision   (damaged | flagged)    %.2f%%" % (100 * flagged.damaged.mean()))
print("base rate   (damaged)              %.2f%%" % (100 * parcels.damaged.mean()))

**1. What it does well and badly.** The scanner **catches 20 of the 24 damaged parcels** - it is a good
detector, and 83% sensitivity is not the problem. What it does badly is the thing that matters
operationally: **of the 52 parcels it flags, 32 are perfectly fine.** Almost two out of every three
diversions would be wasted.

Both facts come from the same place, and it is the reason this is on the assessment: **9% of a large
clean population is a bigger number than 85% of a small damaged one.** 32 false alarms out of 376 clean
parcels beats 20 true catches out of 24 damaged ones, purely on group size. The scanner is not
malfunctioning; the base rate is low.

**2. What the policy costs.** Per 52 flagged parcels, **32 unnecessary inspections** - and it would still
**miss 4 of the 24 damaged parcels**, which continue to the customer exactly as they do today. So the
policy buys 20 catches for 32 wasted inspections and does not close the leak.

**3. The quantity you need.** **The cost of an unnecessary inspection against the cost of a damaged
parcel reaching a customer** - in the same units, whatever those are (euros, minutes, complaints).

B5 cannot settle it because 38.46% is not a recommendation, it is one input to one. If a wasted
inspection costs two minutes and a damaged delivery costs a customer, 32 wasted inspections is trivially
worth 20 catches and you should divert. If inspection means unpacking and repacking a pallet while a van
waits, it is not. **The number is the same in both cases and the decision is opposite**, which is exactly
why a metric is never a decision. That is module 07's subject.

Credit also for: proposing a threshold change if the scanner emits a score rather than a flag; asking
whether damage is worse on some routes so the policy could be targeted; or noting that 24 damaged parcels
is a small sample and the rates carry real uncertainty - a standard error of about 7 percentage points on
the 83%, which is 03-03 applied where it counts.

*Marks: 2 for reading the table correctly, 2 for the cost of the policy including what it still misses,
1 for naming the cost ratio.*

### C3 (4 marks) · Is the model finished?

In [ ]:
distance = parcels.distance_km.to_numpy()
is_north = (parcels.depot == "north").to_numpy().astype(float)
minutes = parcels.minutes.to_numpy()


def fitted_error(*columns):
    design = np.column_stack([np.ones(len(minutes))] + list(columns))
    weights = np.linalg.lstsq(design, minutes, rcond=None)[0]
    return np.mean((minutes - design @ weights) ** 2), weights


only_distance, _ = fitted_error(distance)
with_depot, weights = fitted_error(distance, is_north)
with_weight, _ = fitted_error(distance, is_north, parcels.weight_kg.to_numpy())

print("distance only              MSE %.4f" % only_distance)
print("distance + depot           MSE %.4f   (north adds %.2f minutes)" % (with_depot, weights[2]))
print("distance + depot + weight  MSE %.4f" % with_weight)
print()
truth = 12 + 1.8 * distance + 4.0 * is_north
print("the TRUE generating equation scores  MSE %.4f" % np.mean((minutes - truth) ** 2))
print("the noise that was actually drawn has variance %.4f, not 36.00"
      % np.mean((minutes - truth) ** 2))

**1. Both errors.** Distance alone scores **36.33**; adding `depot` scores **33.77**. The depot
coefficient is **+3.22 minutes** for north, which is a real effect - the data was built with +4.0 and the
fit recovered most of it.

**2. Is it worth having.** A 7% reduction in mean squared error, which sounds small, but the honest
comparison is not against 36.33 - it is against **how much error was ever removable**. In minutes, the
typical miss drops from 6.03 to 5.81. Whether 13 seconds per parcel is worth a column depends on what the
number is for: for a customer-facing delivery estimate, no; for deciding whether the north depot needs
another van, the 3.22 minutes *is* the finding and the error barely matters.

**Marked on being explicit about the comparison**, not on the verdict. "It improved" is not an answer.

**3. How a model beats its own noise.** The expected floor was 36.0, and the two-feature model scored
**33.77** - below it. It scored below the **true generating equation**, which manages only 34.05 on these
rows.

Nothing is wrong. The 400 noise values that were actually drawn have a variance of 34.05 rather than the
36.0 they were drawn *from* - random samples do not hit their parameters exactly, which is 03-02 - and
least squares then bends the line slightly towards the particular noise in these particular rows,
recovering a slope of 1.769 rather than 1.8 and an intercept of 13.17 rather than 12. **Fitting always
captures some of the noise, so the error on the fitting data is always optimistic.**

**4. Adding a column that cannot help.** Adding `weight_kg`, which by construction has no effect on
`minutes`, **lowers the error again** - to 33.766. It cannot possibly improve prediction, and it improves
the measurement anyway, because one more free parameter is one more thing to bend towards the noise.

That is the whole argument for held-out data in three lines, and it is where module 04 starts: **error
measured on the rows you fitted is not evidence.** It can be driven to zero by anyone with enough columns
and no honesty, and it will happily reward a column of pure noise.

*Marks: 1 for both errors and the effect size, 1 for an explicit comparison, 1 for the noise-fitting
explanation, 1 for the weight column and what it demonstrates.*